In [1]:
%pip install scikit-learn

  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.7 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import boto3
import io
import time

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
import sklearn

print("Scikit-learn:", sklearn.__version__)

Scikit-learn: 1.7.2


In [4]:
BUCKET = "trusted-busflow-cb06928c"
PREFIX = "fato_operacao_frota/ano=2026/mes=09/dia=14/"

s3 = boto3.client("s3")

response = s3.list_objects_v2(
    Bucket=BUCKET,
    Prefix=PREFIX
)

arquivos = [
    obj["Key"]
    for obj in response.get("Contents", [])
    if obj["Key"].lower().endswith(".csv")
]

print(f"Quantidade de arquivos: {len(arquivos)}")

for arquivo in arquivos:
    print(arquivo)

Quantidade de arquivos: 17
fato_operacao_frota/ano=2026/mes=09/dia=14/fato_operacao_20260914_090101.csv
fato_operacao_frota/ano=2026/mes=09/dia=14/fato_operacao_20260914_100046.csv
fato_operacao_frota/ano=2026/mes=09/dia=14/fato_operacao_20260914_110105.csv
fato_operacao_frota/ano=2026/mes=09/dia=14/fato_operacao_20260914_120046.csv
fato_operacao_frota/ano=2026/mes=09/dia=14/fato_operacao_20260914_130102.csv
fato_operacao_frota/ano=2026/mes=09/dia=14/fato_operacao_20260914_140133.csv
fato_operacao_frota/ano=2026/mes=09/dia=14/fato_operacao_20260914_150103.csv
fato_operacao_frota/ano=2026/mes=09/dia=14/fato_operacao_20260914_160102.csv
fato_operacao_frota/ano=2026/mes=09/dia=14/fato_operacao_20260914_170219.csv
fato_operacao_frota/ano=2026/mes=09/dia=14/fato_operacao_20260914_180118.csv
fato_operacao_frota/ano=2026/mes=09/dia=14/fato_operacao_20260914_190133.csv
fato_operacao_frota/ano=2026/mes=09/dia=14/fato_operacao_20260914_200200.csv
fato_operacao_frota/ano=2026/mes=09/dia=14/fato_o

In [5]:
arquivo_teste = arquivos[0]

obj = s3.get_object(
    Bucket=BUCKET,
    Key=arquivo_teste
)

df_teste = pd.read_csv(obj["Body"])

In [6]:
print("ARQUIVO:")
print(arquivo_teste)

print("\nCOLUNAS:")
print(df_teste.columns.tolist())

print("\nDIMENSÕES:")
print(df_teste.shape)

print("\nPRIMEIRAS LINHAS:")
display(df_teste.head())

ARQUIVO:
fato_operacao_frota/ano=2026/mes=09/dia=14/fato_operacao_20260914_090101.csv

COLUNAS:
['timestamp_processamento', 'linha_codigo', 'linha_id', 'sentido', 'letreiro_origem', 'letreiro_destino', 'dia_semana', 'hora_minuto', 'frota_ativa_real', 'headway_real_min', 'frota_planejada', 'headway_planejado_min', 'temperatura_c', 'sensacao_termica_c', 'chuva_1h_mm', 'visibilidade_m', 'vento_velocidade_ms', 'clima_evento_id', 'jam_factor', 'velocidade_via_kmh', 'velocidade_freeflow_kmh', 'incidente_critico', 'iac_gargalo_climatico', 'gt_gargalo_trafego', 'hp_horario_pico', 'ac_aderencia_cronograma', 'dfi_demanda_frota_ideal', 'frota_necessaria_estimada', 'deficit_operacional', 'go_gargalo_operacional', 'status_classificacao', 'acao_recomendada']

DIMENSÕES:
(2159, 32)

PRIMEIRAS LINHAS:


,timestamp_processamento,linha_codigo,linha_id,sentido,letreiro_origem,letreiro_destino,dia_semana,hora_minuto,frota_ativa_real,headway_real_min,...,iac_gargalo_climatico,gt_gargalo_trafego,hp_horario_pico,ac_aderencia_cronograma,dfi_demanda_frota_ideal,frota_necessaria_estimada,deficit_operacional,go_gargalo_operacional,status_classificacao,acao_recomendada
0,2026-09-14T09:01:01.896509,208V-10,207,1,TERM. PQ. D. PEDRO II,TERM. A. E. CARVALHO,0,06:00,11,5.5,...,15.9,42.76,0.2,1.000,72.73,16,5,0.3585,Estabilizado,Operação Normal
1,2026-09-14T09:01:01.896509,6840-10,32969,2,TERM. CAPELINHA,TERM. JD. JACIRA,0,06:00,7,8.6,...,15.9,31.06,0.2,1.000,64.29,9,2,0.3081,Estabilizado,Operação Normal
2,2026-09-14T09:01:01.896509,746V-10,33560,2,HOSP. CAMPO LIMPO,JD. REBOUÇAS,0,06:00,5,12.0,...,15.9,32.04,0.2,0.833,70.00,7,2,0.3415,Estabilizado,Operação Normal
3,2026-09-14T09:01:01.896509,609F-10,37,1,TERM. PRINC. ISABEL,CHÁC. SANTANA,0,06:00,6,10.0,...,15.9,48.85,0.2,1.000,66.67,8,2,0.3586,Estabilizado,Operação Normal
4,2026-09-14T09:01:01.896509,5110-10,2209,1,TERM. MERCADO,TERM. SÃO MATEUS,0,06:00,10,6.0,...,15.9,43.32,0.2,1.000,75.00,15,5,0.3656,Estabilizado,Operação Normal


In [7]:
df_teste.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2159 entries, 0 to 2158
Data columns (total 32 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   timestamp_processamento    2159 non-null   object 
 1   linha_codigo               2159 non-null   object 
 2   linha_id                   2159 non-null   int64  
 3   sentido                    2159 non-null   int64  
 4   letreiro_origem            2159 non-null   object 
 5   letreiro_destino           2159 non-null   object 
 6   dia_semana                 2159 non-null   int64  
 7   hora_minuto                2159 non-null   object 
 8   frota_ativa_real           2159 non-null   int64  
 9   headway_real_min           2159 non-null   float64
 10  frota_planejada            2159 non-null   int64  
 11  headway_planejado_min      2159 non-null   float64
 12  temperatura_c              2159 non-null   float64
 13  sensacao_termica_c         2159 non-null   float

In [8]:
dfs = []

for arquivo in arquivos:
    obj = s3.get_object(
        Bucket=BUCKET,
        Key=arquivo
    )

    df = pd.read_csv(obj["Body"])

    df["arquivo_origem"] = arquivo

    dfs.append(df)

df_trusted = pd.concat(dfs, ignore_index=True)

print(f"Total de arquivos: {len(dfs)}")
print(f"Total de registros: {len(df_trusted)}")

Total de arquivos: 17
Total de registros: 35698


In [9]:
# Garantir que o timestamp seja reconhecido como data/hora
df_trusted["timestamp_processamento"] = pd.to_datetime(
    df_trusted["timestamp_processamento"]
)

# Criar resumo dos arquivos
resumo = (
    df_trusted
    .groupby("arquivo_origem")
    .agg(
        timestamp=("timestamp_processamento", "min"),
        registros=("timestamp_processamento", "count")
    )
    .reset_index()
)

# Ordenar cronologicamente
resumo = resumo.sort_values("timestamp").reset_index(drop=True)

# Calcular intervalo entre snapshots
resumo["intervalo_min"] = (
    resumo["timestamp"].diff().dt.total_seconds() / 60
)

# Visualização compacta
resumo[["timestamp", "intervalo_min", "registros"]]

,timestamp,intervalo_min,registros
0,2026-09-14 09:01:01.896509,NaN,2159
1,2026-09-14 10:00:46.851151,59.749244,2184
2,2026-09-14 11:01:05.232951,60.306363,2169
3,2026-09-14 12:00:46.667582,59.690577,2127
4,2026-09-14 13:01:02.465029,60.263291,2068
5,2026-09-14 14:01:33.818099,60.522551,2044
6,2026-09-14 15:01:03.503366,59.494754,2048
7,2026-09-14 16:01:02.933586,59.990504,2048
8,2026-09-14 17:02:19.623198,61.278160,2051
9,2026-09-14 18:01:18.220212,58.976617,2056


In [10]:
df_trusted[
    ["timestamp_processamento", "hora_minuto"]
].drop_duplicates().sort_values("timestamp_processamento").head(30)

,timestamp_processamento,hora_minuto
0,2026-09-14 09:01:01.896509,06:00
2159,2026-09-14 10:00:46.851151,07:00
4343,2026-09-14 11:01:05.232951,08:00
6512,2026-09-14 12:00:46.667582,09:00
8639,2026-09-14 13:01:02.465029,10:00
10707,2026-09-14 14:01:33.818099,11:00
12751,2026-09-14 15:01:03.503366,12:00
14799,2026-09-14 16:01:02.933586,13:00
16847,2026-09-14 17:02:19.623198,14:00
18898,2026-09-14 18:01:18.220212,15:00


In [11]:
df_trusted[
    [
        "timestamp_processamento",
        "hora_minuto",
        "linha_codigo",
        "sentido",
        "go_gargalo_operacional",
        "status_classificacao",
        "acao_recomendada"
    ]
].tail(20)

,timestamp_processamento,hora_minuto,linha_codigo,sentido,go_gargalo_operacional,status_classificacao,acao_recomendada
35678,2026-09-14 23:00:54.301670,20:00,8021-10,1,0.4194,Estabilizado,Operação Normal
35679,2026-09-14 23:00:54.301670,20:00,5107-10,2,0.5090,Estabilizado,Operação Normal
35680,2026-09-14 23:00:54.301670,20:00,3206-10,2,0.4105,Estabilizado,Operação Normal
35681,2026-09-14 23:00:54.301670,20:00,8027-10,1,0.5148,Estabilizado,Operação Normal
35682,2026-09-14 23:00:54.301670,20:00,6813-10,1,0.5227,Estabilizado,Operação Normal
35683,2026-09-14 23:00:54.301670,20:00,6016-10,1,0.6098,Risco,Monitorar Linha em Alerta
35684,2026-09-14 23:00:54.301670,20:00,6232-10,1,0.5215,Estabilizado,Operação Normal
35685,2026-09-14 23:00:54.301670,20:00,TREI-10,2,0.5714,Estabilizado,Operação Normal
35686,2026-09-14 23:00:54.301670,20:00,8007-10,1,0.5129,Estabilizado,Operação Normal
35687,2026-09-14 23:00:54.301670,20:00,9188-42,1,0.5932,Estabilizado,Operação Normal


In [12]:
print(
    df_trusted.groupby("hora_minuto")["linha_codigo"].nunique()
)

hora_minuto
06:00    1137
07:00    1130
08:00    1124
09:00    1099
10:00    1070
11:00    1056
12:00    1051
13:00    1051
14:00    1051
15:00    1057
16:00    1070
17:00    1083
17:30    1086
18:00    1085
18:30    1087
19:00    1089
20:00    1078
Name: linha_codigo, dtype: int64


In [13]:
# Verificar correspondência entre horários consecutivos

pares_horarios = [
    ("17:00", "17:30"),
    ("17:30", "18:00"),
    ("18:00", "18:30"),
    ("18:30", "19:00")
]

for horario_atual, horario_futuro in pares_horarios:

    atual = set(
        zip(
            df_trusted.loc[
                df_trusted["hora_minuto"] == horario_atual,
                "linha_codigo"
            ],
            df_trusted.loc[
                df_trusted["hora_minuto"] == horario_atual,
                "sentido"
            ]
        )
    )

    futuro = set(
        zip(
            df_trusted.loc[
                df_trusted["hora_minuto"] == horario_futuro,
                "linha_codigo"
            ],
            df_trusted.loc[
                df_trusted["hora_minuto"] == horario_futuro,
                "sentido"
            ]
        )
    )

    comuns = atual.intersection(futuro)

    print(
        f"{horario_atual} → {horario_futuro}: "
        f"{len(comuns)} combinações em comum"
    )

17:00 → 17:30: 2082 combinações em comum
17:30 → 18:00: 2092 combinações em comum
18:00 → 18:30: 2096 combinações em comum
18:30 → 19:00: 2092 combinações em comum


In [14]:
# Passo 10 - Criar base de previsão 30 minutos à frente

pares_horarios = [
    ("17:00", "17:30"),
    ("17:30", "18:00"),
    ("18:00", "18:30"),
    ("18:30", "19:00")
]

bases_30min = []

for horario_atual, horario_futuro in pares_horarios:

    atual = df_trusted[
        df_trusted["hora_minuto"] == horario_atual
    ].copy()

    futuro = df_trusted[
        df_trusted["hora_minuto"] == horario_futuro
    ].copy()

    # Renomear o alvo futuro
    futuro = futuro[
        ["linha_codigo", "sentido", "go_gargalo_operacional"]
    ].rename(
        columns={
            "go_gargalo_operacional": "target_30min"
        }
    )

    # Relacionar a operação atual com a mesma linha/sentido no futuro
    base = atual.merge(
        futuro,
        on=["linha_codigo", "sentido"],
        how="inner"
    )

    base["horario_origem"] = horario_atual
    base["horario_alvo"] = horario_futuro

    bases_30min.append(base)

df_30min = pd.concat(
    bases_30min,
    ignore_index=True
)

print("Total de registros:", len(df_30min))
print("Colunas:", len(df_30min.columns))

df_30min[
    [
        "horario_origem",
        "horario_alvo",
        "linha_codigo",
        "sentido",
        "go_gargalo_operacional",
        "target_30min"
    ]
].head(10)

Total de registros: 8362
Colunas: 36


,horario_origem,horario_alvo,linha_codigo,sentido,go_gargalo_operacional,target_30min
0,17:00,17:30,6820-10,2,0.5500,0.5455
1,17:00,17:30,9004-10,2,0.4230,0.4209
2,17:00,17:30,2722-10,1,0.4094,0.3980
3,17:00,17:30,675K-10,2,0.4223,0.4253
4,17:00,17:30,516N-10,1,0.4485,0.4440
5,17:00,17:30,3739-10,2,0.4554,0.4829
6,17:00,17:30,477A-10,1,0.5121,0.4972
7,17:00,17:30,2767-10,1,0.4101,0.4567
8,17:00,17:30,809V-10,2,0.4357,0.4341
9,17:00,17:30,917H-10,1,0.4523,0.5179


In [15]:
features = [
    "linha_codigo",
    "sentido",
    "dia_semana",
    "hora_minuto",
    "frota_ativa_real",
    "headway_real_min",
    "frota_planejada",
    "headway_planejado_min",
    "temperatura_c",
    "sensacao_termica_c",
    "chuva_1h_mm",
    "visibilidade_m",
    "vento_velocidade_ms",
    "jam_factor",
    "velocidade_via_kmh",
    "velocidade_freeflow_kmh",
    "incidente_critico",
    "iac_gargalo_climatico",
    "gt_gargalo_trafego",
    "hp_horario_pico",
    "ac_aderencia_cronograma",
    "dfi_demanda_frota_ideal"
]

target = "target_30min"

print("Variáveis de entrada:", len(features))
print("Alvo:", target)

Variáveis de entrada: 22
Alvo: target_30min


In [16]:
faltantes = [
    coluna for coluna in features
    if coluna not in df_30min.columns
]

print("Variáveis faltantes:", faltantes)

Variáveis faltantes: []


In [17]:
print("Variáveis disponíveis:")
print(features)

Variáveis disponíveis:
['linha_codigo', 'sentido', 'dia_semana', 'hora_minuto', 'frota_ativa_real', 'headway_real_min', 'frota_planejada', 'headway_planejado_min', 'temperatura_c', 'sensacao_termica_c', 'chuva_1h_mm', 'visibilidade_m', 'vento_velocidade_ms', 'jam_factor', 'velocidade_via_kmh', 'velocidade_freeflow_kmh', 'incidente_critico', 'iac_gargalo_climatico', 'gt_gargalo_trafego', 'hp_horario_pico', 'ac_aderencia_cronograma', 'dfi_demanda_frota_ideal']


In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Variáveis categóricas
categoricas = [
    "linha_codigo",
    "hora_minuto"
]

# Variáveis numéricas
numericas = [
    coluna for coluna in features
    if coluna not in categoricas
]

# Pré-processamento
preprocessador = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            categoricas
        ),
        (
            "numericas",
            "passthrough",
            numericas
        )
    ]
)

X = df_30min[features]
y = df_30min[target]

print("X:", X.shape)
print("y:", y.shape)
print("Variáveis categóricas:", categoricas)
print("Variáveis numéricas:", len(numericas))

X: (8362, 22)
y: (8362,)
Variáveis categóricas: ['linha_codigo', 'hora_minuto']
Variáveis numéricas: 20


In [19]:
# Passo 13 - Separação temporal

treino = df_30min[
    df_30min["horario_origem"].isin(["17:00", "17:30"])
].copy()

validacao = df_30min[
    df_30min["horario_origem"].isin(["18:00", "18:30"])
].copy()

X_treino = treino[features]
y_treino = treino[target]

X_validacao = validacao[features]
y_validacao = validacao[target]

print("Treinamento:")
print("X:", X_treino.shape)
print("y:", y_treino.shape)

print("\nValidação:")
print("X:", X_validacao.shape)
print("y:", y_validacao.shape)

print("\nPeríodos de treinamento:")
print(treino["horario_origem"].value_counts().sort_index())

print("\nPeríodos de validação:")
print(validacao["horario_origem"].value_counts().sort_index())

Treinamento:
X: (4174, 22)
y: (4174,)

Validação:
X: (4188, 22)
y: (4188,)

Períodos de treinamento:
horario_origem
17:00    2082
17:30    2092
Name: count, dtype: int64

Períodos de validação:
horario_origem
18:00    2096
18:30    2092
Name: count, dtype: int64


In [20]:
# Passo 14 - Aplicar o pré-processamento

X_treino_processado = preprocessador.fit_transform(X_treino)

X_validacao_processado = preprocessador.transform(X_validacao)

print("X treino processado:", X_treino_processado.shape)
print("X validação processado:", X_validacao_processado.shape)

X treino processado: (4174, 1102)
X validação processado: (4188, 1102)


In [21]:
from sklearn.ensemble import RandomForestRegressor
import time

# Passo 15 - Treinamento do Random Forest

modelo_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

inicio_treino = time.perf_counter()

modelo_rf.fit(
    X_treino_processado,
    y_treino
)

fim_treino = time.perf_counter()

tempo_treino = fim_treino - inicio_treino

print(f"Modelo treinado com sucesso.")
print(f"Tempo de treinamento: {tempo_treino:.4f} segundos")

Modelo treinado com sucesso.
Tempo de treinamento: 3.9030 segundos


In [22]:
# Passo 16 - Previsão dos dados de validação

inicio_previsao = time.perf_counter()

y_pred = modelo_rf.predict(X_validacao_processado)

fim_previsao = time.perf_counter()

tempo_previsao = fim_previsao - inicio_previsao

print(f"Previsões realizadas: {len(y_pred)}")
print(f"Tempo total de previsão: {tempo_previsao:.4f} segundos")
print(f"Tempo médio por registro: {(tempo_previsao / len(y_pred)) * 1000:.4f} ms")

Previsões realizadas: 4188
Tempo total de previsão: 0.0490 segundos
Tempo médio por registro: 0.0117 ms


In [23]:
resultado_validacao = validacao[
    [
        "horario_origem",
        "horario_alvo",
        "linha_codigo",
        "sentido",
        "target_30min"
    ]
].copy()

resultado_validacao["previsto"] = y_pred

resultado_validacao.head(10)

,horario_origem,horario_alvo,linha_codigo,sentido,target_30min,previsto
4174,18:00,18:30,6820-10,1,0.5413,0.499724
4175,18:00,18:30,9004-10,1,0.3765,0.448424
4176,18:00,18:30,2722-10,2,0.3780,0.432134
4177,18:00,18:30,675K-10,2,0.4396,0.451832
4178,18:00,18:30,516N-10,2,0.4444,0.439782
4179,18:00,18:30,3739-10,2,0.4560,0.439623
4180,18:00,18:30,477A-10,1,0.5024,0.496184
4181,18:00,18:30,2767-10,2,0.4464,0.480394
4182,18:00,18:30,809V-10,2,0.3929,0.471003
4183,18:00,18:30,917H-10,2,0.5240,0.451647


In [24]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Métricas de avaliação
mae = mean_absolute_error(y_validacao, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_validacao, y_pred)
)

r2 = r2_score(y_validacao, y_pred)

print(f"MAE:  {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R²:   {r2:.6f}")

MAE:  0.039091
RMSE: 0.049574
R²:   0.272463


In [25]:
print("STATUS:")
print(df_trusted["status_classificacao"].value_counts())

print("\nAÇÕES:")
print(df_trusted["acao_recomendada"].value_counts())

STATUS:
status_classificacao
Estabilizado    35307
Risco             391
Name: count, dtype: int64

AÇÕES:
acao_recomendada
Operação Normal              35307
Monitorar Linha em Alerta      391
Name: count, dtype: int64


In [26]:
print(
    df_trusted[
        ["go_gargalo_operacional", "status_classificacao", "acao_recomendada"]
    ]
    .sort_values("go_gargalo_operacional")
    .drop_duplicates()
    .head(30)
)

      go_gargalo_operacional status_classificacao acao_recomendada
997                   0.2310         Estabilizado  Operação Normal
109                   0.2463         Estabilizado  Operação Normal
1040                  0.2495         Estabilizado  Operação Normal
958                   0.2498         Estabilizado  Operação Normal
829                   0.2505         Estabilizado  Operação Normal
1880                  0.2507         Estabilizado  Operação Normal
1222                  0.2513         Estabilizado  Operação Normal
1565                  0.2515         Estabilizado  Operação Normal
744                   0.2524         Estabilizado  Operação Normal
495                   0.2530         Estabilizado  Operação Normal
48                    0.2535         Estabilizado  Operação Normal
1698                  0.2539         Estabilizado  Operação Normal
1668                  0.2540         Estabilizado  Operação Normal
354                   0.2544         Estabilizado  Operação No

In [27]:
print(
    df_trusted[
        [
            "go_gargalo_operacional",
            "status_classificacao",
            "acao_recomendada"
        ]
    ]
    .sort_values("go_gargalo_operacional", ascending=False)
    .head(50)
)

       go_gargalo_operacional status_classificacao           acao_recomendada
8502                   0.7141                Risco  Monitorar Linha em Alerta
7526                   0.7141                Risco  Monitorar Linha em Alerta
8622                   0.7072                Risco  Monitorar Linha em Alerta
8540                   0.7032                Risco  Monitorar Linha em Alerta
8464                   0.7002                Risco  Monitorar Linha em Alerta
8322                   0.6955                Risco  Monitorar Linha em Alerta
33609                  0.6942                Risco  Monitorar Linha em Alerta
29326                  0.6876                Risco  Monitorar Linha em Alerta
31057                  0.6823                Risco  Monitorar Linha em Alerta
10334                  0.6798                Risco  Monitorar Linha em Alerta
12376                  0.6797                Risco  Monitorar Linha em Alerta
26300                  0.6785                Risco  Monitorar Li

In [28]:
risco = df_trusted[
    df_trusted["status_classificacao"] == "Risco"
]["go_gargalo_operacional"]

normal = df_trusted[
    df_trusted["status_classificacao"] == "Estabilizado"
]["go_gargalo_operacional"]

print("Risco:")
print(f"Mínimo: {risco.min():.4f}")
print(f"Máximo: {risco.max():.4f}")

print("\nEstabilizado:")
print(f"Mínimo: {normal.min():.4f}")
print(f"Máximo: {normal.max():.4f}")

Risco:
Mínimo: 0.6002
Máximo: 0.7141

Estabilizado:
Mínimo: 0.2310
Máximo: 0.5998


In [29]:
print(
    df_trusted.groupby("status_classificacao")["go_gargalo_operacional"]
    .agg(["count", "min", "max", "mean", "median"])
)

                      count     min     max      mean  median
status_classificacao                                         
Estabilizado          35307  0.2310  0.5998  0.426808  0.4243
Risco                   391  0.6002  0.7141  0.627732  0.6235


In [30]:
resultado_validacao["status_real"] = np.where(
    resultado_validacao["target_30min"] >= 0.60,
    "Risco",
    "Estabilizado"
)

resultado_validacao["status_previsto"] = np.where(
    resultado_validacao["previsto"] >= 0.60,
    "Risco",
    "Estabilizado"
)

resultado_validacao.head(10)

,horario_origem,horario_alvo,linha_codigo,sentido,target_30min,previsto,status_real,status_previsto
4174,18:00,18:30,6820-10,1,0.5413,0.499724,Estabilizado,Estabilizado
4175,18:00,18:30,9004-10,1,0.3765,0.448424,Estabilizado,Estabilizado
4176,18:00,18:30,2722-10,2,0.3780,0.432134,Estabilizado,Estabilizado
4177,18:00,18:30,675K-10,2,0.4396,0.451832,Estabilizado,Estabilizado
4178,18:00,18:30,516N-10,2,0.4444,0.439782,Estabilizado,Estabilizado
4179,18:00,18:30,3739-10,2,0.4560,0.439623,Estabilizado,Estabilizado
4180,18:00,18:30,477A-10,1,0.5024,0.496184,Estabilizado,Estabilizado
4181,18:00,18:30,2767-10,2,0.4464,0.480394,Estabilizado,Estabilizado
4182,18:00,18:30,809V-10,2,0.3929,0.471003,Estabilizado,Estabilizado
4183,18:00,18:30,917H-10,2,0.5240,0.451647,Estabilizado,Estabilizado


In [31]:
from sklearn.metrics import confusion_matrix, classification_report

matriz = confusion_matrix(
    resultado_validacao["status_real"],
    resultado_validacao["status_previsto"],
    labels=["Estabilizado", "Risco"]
)

print("Matriz de confusão:")
print(matriz)

print("\nRelatório de classificação:")
print(
    classification_report(
        resultado_validacao["status_real"],
        resultado_validacao["status_previsto"],
        labels=["Estabilizado", "Risco"],
        zero_division=0
    )
)

Matriz de confusão:
[[4098    0]
 [  90    0]]

Relatório de classificação:
              precision    recall  f1-score   support

Estabilizado       0.98      1.00      0.99      4098
       Risco       0.00      0.00      0.00        90

    accuracy                           0.98      4188
   macro avg       0.49      0.50      0.49      4188
weighted avg       0.96      0.98      0.97      4188



In [32]:
risco_real = (
    resultado_validacao["status_real"] == "Risco"
)

risco_previsto = (
    resultado_validacao["status_previsto"] == "Risco"
)

print("Riscos reais:", risco_real.sum())
print("Riscos previstos:", risco_previsto.sum())

Riscos reais: 90
Riscos previstos: 0


In [33]:
riscos_validacao = resultado_validacao[
    resultado_validacao["status_real"] == "Risco"
].copy()

print("Quantidade de riscos:", len(riscos_validacao))

riscos_validacao[
    [
        "horario_origem",
        "horario_alvo",
        "linha_codigo",
        "sentido",
        "target_30min",
        "previsto"
    ]
].head(20)

Quantidade de riscos: 90


,horario_origem,horario_alvo,linha_codigo,sentido,target_30min,previsto
4190,18:00,18:30,3064-41,2,0.6100,0.527623
4311,18:00,18:30,5391-22,2,0.6334,0.485784
4416,18:00,18:30,5630-21,2,0.6422,0.531535
4418,18:00,18:30,3015-10,1,0.6164,0.516966
4446,18:00,18:30,3722-10,1,0.6138,0.484306
4499,18:00,18:30,637G-51,1,0.6325,0.524777
4579,18:00,18:30,2104-10,2,0.6557,0.502408
4707,18:00,18:30,2522-10,2,0.6823,0.481447
4791,18:00,18:30,3728-41,2,0.6189,0.454822
4918,18:00,18:30,7057-10,1,0.6154,0.526020


In [34]:
print(
    riscos_validacao["horario_origem"]
    .value_counts()
    .sort_index()
)

horario_origem
18:00    55
18:30    35
Name: count, dtype: int64


In [35]:
print(
    riscos_validacao["linha_codigo"]
    .value_counts()
    .head(20)
)

linha_codigo
3064-41    2
2104-10    2
6012-31    2
TREI-10    2
8252-10    2
6806-10    2
4007-31    2
6818-10    2
8600-22    2
3013-10    2
3004-10    2
909T-1     2
3459-23    2
5100-10    2
1022-10    2
8015-21    2
5630-21    1
5391-22    1
7057-10    1
3728-41    1
Name: count, dtype: int64


In [36]:
variaveis_analise = [
    "frota_ativa_real",
    "headway_real_min",
    "frota_planejada",
    "headway_planejado_min",
    "temperatura_c",
    "sensacao_termica_c",
    "chuva_1h_mm",
    "visibilidade_m",
    "vento_velocidade_ms",
    "jam_factor",
    "velocidade_via_kmh",
    "velocidade_freeflow_kmh",
    "iac_gargalo_climatico",
    "gt_gargalo_trafego",
    "hp_horario_pico",
    "ac_aderencia_cronograma",
    "dfi_demanda_frota_ideal"
]

comparacao = df_30min.copy()

comparacao["classe"] = np.where(
    comparacao["target_30min"] >= 0.60,
    "Risco",
    "Estabilizado"
)

resumo_variaveis = (
    comparacao
    .groupby("classe")[variaveis_analise]
    .mean()
    .T
)

print(resumo_variaveis)

classe                   Estabilizado        Risco
frota_ativa_real             5.440463     1.611465
headway_real_min            17.476222    44.299363
frota_planejada              5.569653     1.611465
headway_planejado_min       20.000000    20.000000
temperatura_c               14.462540    14.453758
sensacao_termica_c          14.409989    14.400764
chuva_1h_mm                  0.445325     0.427580
visibilidade_m            7479.058135  7673.070064
vento_velocidade_ms          2.699287     2.709108
jam_factor                   4.575974     4.948790
velocidade_via_kmh          17.518060    17.711975
velocidade_freeflow_kmh     32.020606    33.409554
iac_gargalo_climatico       19.280417    18.861401
gt_gargalo_trafego          42.691480    43.351720
hp_horario_pico              0.800000     0.800000
ac_aderencia_cronograma      0.909100     0.530739
dfi_demanda_frota_ideal     73.870544    87.844968


In [37]:
riscos_validacao["erro_absoluto"] = (
    riscos_validacao["target_30min"] -
    riscos_validacao["previsto"]
).abs()

print(
    riscos_validacao[
        [
            "linha_codigo",
            "sentido",
            "target_30min",
            "previsto",
            "erro_absoluto"
        ]
    ]
    .sort_values("target_30min", ascending=False)
    .head(20)
)

     linha_codigo  sentido  target_30min  previsto  erro_absoluto
8032      805L-10        1        0.6942  0.511842       0.182358
4707      2522-10        2        0.6823  0.481447       0.200853
5443      809J-10        2        0.6768  0.502979       0.173821
6110      8065-10        2        0.6767  0.525834       0.150866
8301      8010-21        2        0.6742  0.537779       0.136421
5790      5131-10        1        0.6693  0.495062       0.174238
6244      8015-21        2        0.6669  0.553504       0.113396
8216      2104-10        1        0.6636  0.505377       0.158223
8343      8015-21        2        0.6628  0.547537       0.115263
8196      1022-10        2        0.6594  0.529488       0.129912
6644      6403-10        1        0.6586  0.510806       0.147794
5810      3139-10        2        0.6585  0.496710       0.161790
4579      2104-10        2        0.6557  0.502408       0.153292
7692      8600-22        2        0.6518  0.581414       0.070386
8241      

In [38]:
variaveis_criticas = [
    "frota_ativa_real",
    "headway_real_min",
    "frota_planejada",
    "headway_planejado_min",
    "ac_aderencia_cronograma",
    "dfi_demanda_frota_ideal",
    "jam_factor",
    "gt_gargalo_trafego",
    "incidente_critico"
]

chaves = [
    "horario_origem",
    "horario_alvo",
    "linha_codigo",
    "sentido"
]

dados_riscos = riscos_validacao.merge(
    df_30min[
        chaves + variaveis_criticas
    ],
    on=chaves,
    how="left"
)

print("Registros:", len(dados_riscos))
print("Valores ausentes:")
print(dados_riscos[variaveis_criticas].isna().sum())

Registros: 90
Valores ausentes:
frota_ativa_real           0
headway_real_min           0
frota_planejada            0
headway_planejado_min      0
ac_aderencia_cronograma    0
dfi_demanda_frota_ideal    0
jam_factor                 0
gt_gargalo_trafego         0
incidente_critico          0
dtype: int64


In [39]:
print(
    dados_riscos[
        ["linha_codigo", "sentido", "target_30min", "previsto"]
        + variaveis_criticas
    ]
    .sort_values("target_30min", ascending=False)
    .head(30)
)

   linha_codigo  sentido  target_30min  previsto  frota_ativa_real  \
75      805L-10        1        0.6942  0.511842                 2   
7       2522-10        2        0.6823  0.481447                 2   
14      809J-10        2        0.6768  0.502979                 2   
37      8065-10        2        0.6767  0.525834                 1   
84      8010-21        2        0.6742  0.537779                 1   
21      5131-10        1        0.6693  0.495062                 2   
51      8015-21        2        0.6669  0.553504                 1   
79      2104-10        1        0.6636  0.505377                 2   
88      8015-21        2        0.6628  0.547537                 1   
78      1022-10        2        0.6594  0.529488                 1   
58      6403-10        1        0.6586  0.510806                 3   
22      3139-10        2        0.6585  0.496710                 2   
6       2104-10        2        0.6557  0.502408                 4   
67      8600-22     

In [40]:
riscos_validacao["erro_absoluto"] = (
    riscos_validacao["target_30min"] -
    riscos_validacao["previsto"]
).abs()

riscos_validacao["faixa_risco"] = pd.cut(
    riscos_validacao["target_30min"],
    bins=[0.60, 0.62, 0.65, 0.70, 1.00],
    labels=["0,60–0,62", "0,62–0,65", "0,65–0,70", "≥ 0,70"],
    include_lowest=True
)

analise_faixas = (
    riscos_validacao
    .groupby("faixa_risco", observed=False)
    .agg(
        casos=("target_30min", "count"),
        target_medio=("target_30min", "mean"),
        previsto_medio=("previsto", "mean"),
        erro_medio=("erro_absoluto", "mean")
    )
    .reset_index()
)

print(analise_faixas)

  faixa_risco  casos  target_medio  previsto_medio  erro_medio
0   0,60–0,62     36      0.608908        0.506140    0.102768
1   0,62–0,65     36      0.631667        0.510741    0.120926
2   0,65–0,70     18      0.664111        0.527548    0.136563
3      ≥ 0,70      0           NaN             NaN         NaN


In [41]:
comparacao_grupos = (
    dados_riscos.groupby(
        np.where(
            dados_riscos["target_30min"] >= 0.60,
            "Risco",
            "Estabilizado"
        )
    )[variaveis_criticas]
    .agg(["mean", "median"])
)

print(comparacao_grupos)

      frota_ativa_real        headway_real_min        frota_planejada         \
                  mean median             mean median            mean median   
Risco         1.622222    1.0        44.055556   60.0        1.622222    1.0   

      headway_planejado_min        ac_aderencia_cronograma         \
                       mean median                    mean median   
Risco                  20.0   20.0                0.536989  0.333   

      dfi_demanda_frota_ideal        jam_factor        gt_gargalo_trafego  \
                         mean median       mean median               mean   
Risco               87.777778  100.0   5.137556   4.92             44.574   

              incidente_critico         
       median              mean median  
Risco  46.495          0.244444    0.0  


In [42]:
# Criar classificação real dos registros
dados_comparacao = dados_riscos.copy()

dados_comparacao["classe"] = np.where(
    dados_comparacao["target_30min"] >= 0.60,
    "Risco",
    "Estabilizado"
)

# Comparação entre os grupos
comparacao = (
    dados_comparacao
    .groupby("classe")[variaveis_criticas]
    .agg(["mean", "median"])
)

print(comparacao)

       frota_ativa_real        headway_real_min        frota_planejada         \
                   mean median             mean median            mean median   
classe                                                                          
Risco          1.622222    1.0        44.055556   60.0        1.622222    1.0   

       headway_planejado_min        ac_aderencia_cronograma         \
                        mean median                    mean median   
classe                                                               
Risco                   20.0   20.0                0.536989  0.333   

       dfi_demanda_frota_ideal        jam_factor        gt_gargalo_trafego  \
                          mean median       mean median               mean   
classe                                                                       
Risco                87.777778  100.0   5.137556   4.92             44.574   

               incidente_critico         
        median              mean median

In [43]:
# Criar classificação real para todos os registros
df_comparacao = df_30min.copy()

df_comparacao["classe"] = np.where(
    df_comparacao["target_30min"] >= 0.60,
    "Risco",
    "Estabilizado"
)

print(df_comparacao["classe"].value_counts())

classe
Estabilizado    8205
Risco            157
Name: count, dtype: int64


In [44]:
medias = (
    df_comparacao
    .groupby("classe")[variaveis_criticas]
    .mean()
    .T
)

print(medias)

classe                   Estabilizado      Risco
frota_ativa_real             5.440463   1.611465
headway_real_min            17.476222  44.299363
frota_planejada              5.569653   1.611465
headway_planejado_min       20.000000  20.000000
ac_aderencia_cronograma      0.909100   0.530739
dfi_demanda_frota_ideal     73.870544  87.844968
jam_factor                   4.575974   4.948790
gt_gargalo_trafego          42.691480  43.351720
incidente_critico            0.327361   0.248408


In [45]:
medias["diferenca"] = (
    medias["Risco"] - medias["Estabilizado"]
)

medias["diferenca_percentual"] = (
    (medias["Risco"] - medias["Estabilizado"])
    / medias["Estabilizado"].replace(0, np.nan)
) * 100

print(
    medias.sort_values(
        "diferenca_percentual",
        ascending=False
    )
)

classe                   Estabilizado      Risco  diferenca  \
headway_real_min            17.476222  44.299363  26.823141   
dfi_demanda_frota_ideal     73.870544  87.844968  13.974425   
jam_factor                   4.575974   4.948790   0.372815   
gt_gargalo_trafego          42.691480  43.351720   0.660240   
headway_planejado_min       20.000000  20.000000   0.000000   
incidente_critico            0.327361   0.248408  -0.078954   
ac_aderencia_cronograma      0.909100   0.530739  -0.378361   
frota_ativa_real             5.440463   1.611465  -3.828998   
frota_planejada              5.569653   1.611465  -3.958188   

classe                   diferenca_percentual  
headway_real_min                   153.483639  
dfi_demanda_frota_ideal             18.917452  
jam_factor                           8.147235  
gt_gargalo_trafego                   1.546538  
headway_planejado_min                0.000000  
incidente_critico                  -24.118216  
ac_aderencia_cronograma          

In [46]:
import pandas as pd

# Recuperar os nomes das variáveis após o One-Hot Encoding
nomes_features = preprocessador.get_feature_names_out()

importancias = pd.DataFrame({
    "feature": nomes_features,
    "importancia": modelo_rf.feature_importances_
})

importancias = (
    importancias
    .sort_values("importancia", ascending=False)
    .reset_index(drop=True)
)

print(importancias.head(30))

                               feature  importancia
0   numericas__dfi_demanda_frota_ideal     0.366348
1         numericas__incidente_critico     0.100118
2        numericas__gt_gargalo_trafego     0.045210
3   numericas__velocidade_freeflow_kmh     0.040497
4        numericas__velocidade_via_kmh     0.039044
5                numericas__jam_factor     0.033547
6          numericas__headway_real_min     0.032688
7          numericas__frota_ativa_real     0.020787
8           numericas__frota_planejada     0.020256
9    categoricas__linha_codigo_2017-10     0.007111
10            numericas__temperatura_c     0.006059
11      categoricas__hora_minuto_17:30     0.005823
12      numericas__vento_velocidade_ms     0.004953
13       numericas__sensacao_termica_c     0.004926
14      categoricas__hora_minuto_17:00     0.004906
15              numericas__chuva_1h_mm     0.004819
16  numericas__ac_aderencia_cronograma     0.004356
17   categoricas__linha_codigo_6039-21     0.004296
18   categor

In [47]:
subindices_derivados = [
    "iac_gargalo_climatico",
    "gt_gargalo_trafego",
    "hp_horario_pico",
    "ac_aderencia_cronograma",
    "dfi_demanda_frota_ideal"
]

In [48]:
# ============================================================
# MODELO B — RANDOM FOREST COM FEATURES PRIMITIVAS
# ============================================================

features_modelo_b = [
    coluna for coluna in features
    if coluna not in [
        "iac_gargalo_climatico",
        "gt_gargalo_trafego",
        "hp_horario_pico",
        "ac_aderencia_cronograma",
        "dfi_demanda_frota_ideal"
    ]
]

print("Variáveis do Modelo B:", len(features_modelo_b))
print(features_modelo_b)

Variáveis do Modelo B: 17
['linha_codigo', 'sentido', 'dia_semana', 'hora_minuto', 'frota_ativa_real', 'headway_real_min', 'frota_planejada', 'headway_planejado_min', 'temperatura_c', 'sensacao_termica_c', 'chuva_1h_mm', 'visibilidade_m', 'vento_velocidade_ms', 'jam_factor', 'velocidade_via_kmh', 'velocidade_freeflow_kmh', 'incidente_critico']


In [49]:
# ============================================================
# MODELO B — SEPARAÇÃO DAS VARIÁVEIS
# ============================================================

categoricas_b = [
    "linha_codigo",
    "hora_minuto"
]

numericas_b = [
    coluna for coluna in features_modelo_b
    if coluna not in categoricas_b
]

print("Variáveis categóricas:", categoricas_b)
print("Quantidade:", len(categoricas_b))

print("\nVariáveis numéricas:", numericas_b)
print("Quantidade:", len(numericas_b))

Variáveis categóricas: ['linha_codigo', 'hora_minuto']
Quantidade: 2

Variáveis numéricas: ['sentido', 'dia_semana', 'frota_ativa_real', 'headway_real_min', 'frota_planejada', 'headway_planejado_min', 'temperatura_c', 'sensacao_termica_c', 'chuva_1h_mm', 'visibilidade_m', 'vento_velocidade_ms', 'jam_factor', 'velocidade_via_kmh', 'velocidade_freeflow_kmh', 'incidente_critico']
Quantidade: 15


In [50]:
# ============================================================
# MODELO B — PREPROCESSAMENTO
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessador_b = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            categoricas_b
        ),
        (
            "numericas",
            "passthrough",
            numericas_b
        )
    ]
)

X_b = df_30min[features_modelo_b]
y_b = df_30min[target]

print("X Modelo B:", X_b.shape)
print("y Modelo B:", y_b.shape)

X Modelo B: (8362, 17)
y Modelo B: (8362,)


In [51]:
# ============================================================
# MODELO B — DIVISÃO TEMPORAL
# ============================================================

treino_b = df_30min[
    df_30min["horario_origem"].isin(["17:00", "17:30"])
].copy()

validacao_b = df_30min[
    df_30min["horario_origem"].isin(["18:00", "18:30"])
].copy()

X_treino_b = treino_b[features_modelo_b]
y_treino_b = treino_b[target]

X_validacao_b = validacao_b[features_modelo_b]
y_validacao_b = validacao_b[target]

print("Treino X:", X_treino_b.shape)
print("Treino y:", y_treino_b.shape)

print("Validação X:", X_validacao_b.shape)
print("Validação y:", y_validacao_b.shape)

print("\nPeríodos de treino:")
print(treino_b["horario_origem"].value_counts().sort_index())

print("\nPeríodos de validação:")
print(validacao_b["horario_origem"].value_counts().sort_index())

Treino X: (4174, 17)
Treino y: (4174,)
Validação X: (4188, 17)
Validação y: (4188,)

Períodos de treino:
horario_origem
17:00    2082
17:30    2092
Name: count, dtype: int64

Períodos de validação:
horario_origem
18:00    2096
18:30    2092
Name: count, dtype: int64


In [52]:
# ============================================================
# MODELO B — APLICAÇÃO DO PREPROCESSAMENTO
# ============================================================

X_treino_processado_b = preprocessador_b.fit_transform(X_treino_b)
X_validacao_processado_b = preprocessador_b.transform(X_validacao_b)

print("X treino processado:", X_treino_processado_b.shape)
print("X validação processado:", X_validacao_processado_b.shape)

X treino processado: (4174, 1097)
X validação processado: (4188, 1097)


In [53]:
# ============================================================
# MODELO B — RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestRegressor
import time

modelo_rf_b = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

inicio_treino_b = time.perf_counter()

modelo_rf_b.fit(
    X_treino_processado_b,
    y_treino_b
)

fim_treino_b = time.perf_counter()

tempo_treino_b = fim_treino_b - inicio_treino_b

print("Modelo B treinado com sucesso.")
print(f"Tempo de treinamento: {tempo_treino_b:.4f} segundos")

Modelo B treinado com sucesso.
Tempo de treinamento: 3.0042 segundos


In [54]:
# ============================================================
# MODELO B — PREVISÃO
# ============================================================

inicio_previsao_b = time.perf_counter()

y_pred_b = modelo_rf_b.predict(
    X_validacao_processado_b
)

fim_previsao_b = time.perf_counter()

tempo_previsao_b = fim_previsao_b - inicio_previsao_b

print(f"Previsões realizadas: {len(y_pred_b)}")
print(f"Tempo total de previsão: {tempo_previsao_b:.4f} segundos")
print(
    f"Tempo médio por registro: "
    f"{(tempo_previsao_b / len(y_pred_b)) * 1000:.4f} ms"
)

Previsões realizadas: 4188
Tempo total de previsão: 0.0485 segundos
Tempo médio por registro: 0.0116 ms


In [55]:
# ============================================================
# MODELO B — RESULTADOS
# ============================================================

resultado_validacao_b = validacao_b[
    [
        "horario_origem",
        "horario_alvo",
        "linha_codigo",
        "sentido",
        "target_30min"
    ]
].copy()

resultado_validacao_b["previsto"] = y_pred_b

print("Total de registros:", len(resultado_validacao_b))

resultado_validacao_b.head(10)

Total de registros: 4188


,horario_origem,horario_alvo,linha_codigo,sentido,target_30min,previsto
4174,18:00,18:30,6820-10,1,0.5413,0.500196
4175,18:00,18:30,9004-10,1,0.3765,0.442326
4176,18:00,18:30,2722-10,2,0.3780,0.435053
4177,18:00,18:30,675K-10,2,0.4396,0.448156
4178,18:00,18:30,516N-10,2,0.4444,0.438493
4179,18:00,18:30,3739-10,2,0.4560,0.441899
4180,18:00,18:30,477A-10,1,0.5024,0.499910
4181,18:00,18:30,2767-10,2,0.4464,0.483575
4182,18:00,18:30,809V-10,2,0.3929,0.459741
4183,18:00,18:30,917H-10,2,0.5240,0.452421


In [56]:
# ============================================================
# MODELO B — MÉTRICAS DE REGRESSÃO
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_b = mean_absolute_error(
    y_validacao_b,
    y_pred_b
)

rmse_b = np.sqrt(
    mean_squared_error(
        y_validacao_b,
        y_pred_b
    )
)

r2_b = r2_score(
    y_validacao_b,
    y_pred_b
)

print(f"MAE:  {mae_b:.6f}")
print(f"RMSE: {rmse_b:.6f}")
print(f"R²:   {r2_b:.6f}")

MAE:  0.039028
RMSE: 0.049504
R²:   0.274513


In [57]:
# ============================================================
# MODELO B — CLASSIFICAÇÃO OPERACIONAL
# ============================================================

resultado_validacao_b["status_real"] = np.where(
    resultado_validacao_b["target_30min"] >= 0.60,
    "Risco",
    "Estabilizado"
)

resultado_validacao_b["status_previsto"] = np.where(
    resultado_validacao_b["previsto"] >= 0.60,
    "Risco",
    "Estabilizado"
)

print("Distribuição REAL:")
print(resultado_validacao_b["status_real"].value_counts())

print("\nDistribuição PREVISTA:")
print(resultado_validacao_b["status_previsto"].value_counts())

Distribuição REAL:
status_real
Estabilizado    4098
Risco             90
Name: count, dtype: int64

Distribuição PREVISTA:
status_previsto
Estabilizado    4188
Name: count, dtype: int64


In [58]:
# ============================================================
# MODELO B — MATRIZ DE CONFUSÃO
# ============================================================

from sklearn.metrics import confusion_matrix, classification_report

matriz_b = confusion_matrix(
    resultado_validacao_b["status_real"],
    resultado_validacao_b["status_previsto"],
    labels=["Estabilizado", "Risco"]
)

print("Matriz de confusão:")
print(matriz_b)

print("\nClassification Report:")
print(
    classification_report(
        resultado_validacao_b["status_real"],
        resultado_validacao_b["status_previsto"],
        labels=["Estabilizado", "Risco"],
        zero_division=0
    )
)

Matriz de confusão:
[[4098    0]
 [  90    0]]

Classification Report:
              precision    recall  f1-score   support

Estabilizado       0.98      1.00      0.99      4098
       Risco       0.00      0.00      0.00        90

    accuracy                           0.98      4188
   macro avg       0.49      0.50      0.49      4188
weighted avg       0.96      0.98      0.97      4188



In [59]:
# ============================================================
# MODELO B — ANÁLISE DOS CASOS DE RISCO
# ============================================================

riscos_validacao_b = resultado_validacao_b[
    resultado_validacao_b["status_real"] == "Risco"
].copy()

riscos_validacao_b["erro_absoluto"] = (
    riscos_validacao_b["target_30min"]
    - riscos_validacao_b["previsto"]
).abs()

print("Total de riscos reais:", len(riscos_validacao_b))

print("\nMédia do alvo real:")
print(riscos_validacao_b["target_30min"].mean())

print("\nMédia da previsão:")
print(riscos_validacao_b["previsto"].mean())

print("\nMaior valor real:")
print(riscos_validacao_b["target_30min"].max())

print("\nMaior previsão:")
print(riscos_validacao_b["previsto"].max())

print("\nErro absoluto médio nos casos de risco:")
print(riscos_validacao_b["erro_absoluto"].mean())

Total de riscos reais: 90

Média do alvo real:
0.6290522222222221

Média da previsão:
0.5152135410220854

Maior valor real:
0.6942

Maior previsão:
0.5932051476464801

Erro absoluto médio nos casos de risco:
0.11383868120013682


In [60]:
# ============================================================
# MODELO B — FAIXAS DE RISCO
# ============================================================

riscos_validacao_b["faixa_risco"] = pd.cut(
    riscos_validacao_b["target_30min"],
    bins=[0.60, 0.62, 0.65, 0.70, float("inf")],
    labels=["0,60–0,62", "0,62–0,65", "0,65–0,70", "≥ 0,70"],
    right=False
)

faixas_risco_b = (
    riscos_validacao_b
    .groupby("faixa_risco", observed=False)
    .agg(
        casos=("target_30min", "count"),
        target_medio=("target_30min", "mean"),
        previsto_medio=("previsto", "mean"),
        erro_medio=("erro_absoluto", "mean")
    )
    .reset_index()
)

faixas_risco_b

,faixa_risco,casos,target_medio,previsto_medio,erro_medio
0,"0,60–0,62",36,0.608908,0.507181,0.101728
1,"0,62–0,65",36,0.631667,0.514222,0.117445
2,"0,65–0,70",18,0.664111,0.533263,0.130848
3,"≥ 0,70",0,NaN,NaN,NaN


In [61]:
# ============================================================
# MODELO B — IMPORTÂNCIA DAS FEATURES
# ============================================================

nomes_features_b = preprocessador_b.get_feature_names_out()

importancias_b = pd.DataFrame({
    "feature": nomes_features_b,
    "importancia": modelo_rf_b.feature_importances_
}).sort_values(
    "importancia",
    ascending=False
).reset_index(drop=True)

importancias_b.head(20)

,feature,importancia
0,numericas__incidente_critico,0.256391
1,numericas__headway_real_min,0.153907
2,numericas__jam_factor,0.079050
3,numericas__frota_planejada,0.068859
4,numericas__frota_ativa_real,0.066686
5,numericas__velocidade_freeflow_kmh,0.043633
6,numericas__velocidade_via_kmh,0.037958
7,categoricas__linha_codigo_2017-10,0.007308
8,categoricas__hora_minuto_17:00,0.006584
9,numericas__temperatura_c,0.006393


In [62]:
# ============================================================
# COMPARAÇÃO MODELO A × MODELO B
# ============================================================

comparacao_ab = pd.DataFrame({
    "Métrica": [
        "MAE",
        "RMSE",
        "R²",
        "Tempo treinamento (s)",
        "Tempo previsão (s)",
        "Tempo previsão por registro (ms)",
        "Riscos reais",
        "Riscos previstos",
        "Recall - Risco"
    ],
    "Modelo A": [
        mae,
        rmse,
        r2,
        tempo_treino,
        tempo_previsao,
        (tempo_previsao / len(y_pred)) * 1000,
        90,
        0,
        0.00
    ],
    "Modelo B": [
        mae_b,
        rmse_b,
        r2_b,
        tempo_treino_b,
        tempo_previsao_b,
        (tempo_previsao_b / len(y_pred_b)) * 1000,
        90,
        0,
        0.00
    ]
})

comparacao_ab

,Métrica,Modelo A,Modelo B
0,MAE,0.039091,0.039028
1,RMSE,0.049574,0.049504
2,R²,0.272463,0.274513
3,Tempo treinamento (s),3.902969,3.004195
4,Tempo previsão (s),0.048956,0.048508
5,Tempo previsão por registro (ms),0.011689,0.011583
6,Riscos reais,90.000000,90.000000
7,Riscos previstos,0.000000,0.000000
8,Recall - Risco,0.000000,0.000000


In [63]:
# ============================================================
# MODELO B2 — DISTRIBUIÇÃO DOS CASOS NO TREINO
# ============================================================

import numpy as np

status_treino_b2 = np.where(
    y_treino_b >= 0.60,
    "Risco",
    "Estabilizado"
)

valores, contagens = np.unique(
    status_treino_b2,
    return_counts=True
)

print("Distribuição no conjunto de treino:")

for valor, contagem in zip(valores, contagens):
    print(f"{valor}: {contagem}")

total_treino_b2 = len(y_treino_b)
riscos_treino_b2 = np.sum(y_treino_b >= 0.60)

print(f"\nTotal: {total_treino_b2}")
print(f"Riscos: {riscos_treino_b2}")
print(
    f"Percentual de risco: "
    f"{(riscos_treino_b2 / total_treino_b2) * 100:.2f}%"
)

Distribuição no conjunto de treino:
Estabilizado: 4107
Risco: 67

Total: 4174
Riscos: 67
Percentual de risco: 1.61%


In [64]:
# ============================================================
# MODELO B2 — PESOS PARA CASOS DE RISCO
# ============================================================

pesos_treino_b2 = np.where(
    y_treino_b >= 0.60,
    5.0,   # casos de Risco
    1.0    # casos Estabilizados
)

print("Quantidade de registros:", len(pesos_treino_b2))

print(
    "Peso 1 (Estabilizado):",
    np.sum(pesos_treino_b2 == 1.0)
)

print(
    "Peso 5 (Risco):",
    np.sum(pesos_treino_b2 == 5.0)
)

print(
    "Peso médio:",
    pesos_treino_b2.mean()
)

Quantidade de registros: 4174
Peso 1 (Estabilizado): 4107
Peso 5 (Risco): 67
Peso médio: 1.06420699568759


In [65]:
# ============================================================
# MODELO B2 — RANDOM FOREST PONDERADO
# ============================================================

from sklearn.ensemble import RandomForestRegressor
import time

modelo_rf_b2 = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

inicio_treino_b2 = time.perf_counter()

modelo_rf_b2.fit(
    X_treino_processado_b,
    y_treino_b,
    sample_weight=pesos_treino_b2
)

fim_treino_b2 = time.perf_counter()

tempo_treino_b2 = fim_treino_b2 - inicio_treino_b2

print("Modelo B2 treinado com sucesso.")
print(f"Tempo de treinamento: {tempo_treino_b2:.4f} segundos")

Modelo B2 treinado com sucesso.
Tempo de treinamento: 5.8098 segundos


In [66]:
# ============================================================
# MODELO B2 — PREVISÃO
# ============================================================

inicio_previsao_b2 = time.perf_counter()

y_pred_b2 = modelo_rf_b2.predict(
    X_validacao_processado_b
)

fim_previsao_b2 = time.perf_counter()

tempo_previsao_b2 = fim_previsao_b2 - inicio_previsao_b2

print(f"Previsões realizadas: {len(y_pred_b2)}")
print(f"Tempo total de previsão: {tempo_previsao_b2:.4f} segundos")
print(
    f"Tempo médio por registro: "
    f"{(tempo_previsao_b2 / len(y_pred_b2)) * 1000:.4f} ms"
)

print(f"\nMenor previsão: {y_pred_b2.min():.6f}")
print(f"Média das previsões: {y_pred_b2.mean():.6f}")
print(f"Maior previsão: {y_pred_b2.max():.6f}")

Previsões realizadas: 4188
Tempo total de previsão: 0.0468 segundos
Tempo médio por registro: 0.0112 ms

Menor previsão: 0.419992
Média das previsões: 0.474374
Maior previsão: 0.606251


In [67]:
# ============================================================
# MODELO B2 — MÉTRICAS E CLASSIFICAÇÃO DE RISCO
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    confusion_matrix,
    classification_report
)
import numpy as np

# -------------------------
# Métricas de regressão
# -------------------------

mae_b2 = mean_absolute_error(
    y_validacao_b,
    y_pred_b2
)

rmse_b2 = np.sqrt(
    mean_squared_error(
        y_validacao_b,
        y_pred_b2
    )
)

r2_b2 = r2_score(
    y_validacao_b,
    y_pred_b2
)

print("MÉTRICAS DE REGRESSÃO")
print(f"MAE:  {mae_b2:.6f}")
print(f"RMSE: {rmse_b2:.6f}")
print(f"R²:   {r2_b2:.6f}")


# -------------------------
# Tabela de resultados
# -------------------------

resultado_validacao_b2 = validacao_b[
    [
        "horario_origem",
        "horario_alvo",
        "linha_codigo",
        "sentido",
        "target_30min"
    ]
].copy()

resultado_validacao_b2["previsto"] = y_pred_b2


# -------------------------
# Classificação operacional
# -------------------------

resultado_validacao_b2["status_real"] = np.where(
    resultado_validacao_b2["target_30min"] >= 0.60,
    "Risco",
    "Estabilizado"
)

resultado_validacao_b2["status_previsto"] = np.where(
    resultado_validacao_b2["previsto"] >= 0.60,
    "Risco",
    "Estabilizado"
)

print("\nDISTRIBUIÇÃO PREVISTA")
print(
    resultado_validacao_b2[
        "status_previsto"
    ].value_counts()
)


# -------------------------
# Matriz de confusão
# -------------------------

matriz_b2 = confusion_matrix(
    resultado_validacao_b2["status_real"],
    resultado_validacao_b2["status_previsto"],
    labels=["Estabilizado", "Risco"]
)

print("\nMATRIZ DE CONFUSÃO")
print(matriz_b2)


# -------------------------
# Classification report
# -------------------------

print("\nCLASSIFICATION REPORT")

print(
    classification_report(
        resultado_validacao_b2["status_real"],
        resultado_validacao_b2["status_previsto"],
        labels=["Estabilizado", "Risco"],
        zero_division=0
    )
)

MÉTRICAS DE REGRESSÃO
MAE:  0.039341
RMSE: 0.049892
R²:   0.263084

DISTRIBUIÇÃO PREVISTA
status_previsto
Estabilizado    4187
Risco              1
Name: count, dtype: int64

MATRIZ DE CONFUSÃO
[[4097    1]
 [  90    0]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

Estabilizado       0.98      1.00      0.99      4098
       Risco       0.00      0.00      0.00        90

    accuracy                           0.98      4188
   macro avg       0.49      0.50      0.49      4188
weighted avg       0.96      0.98      0.97      4188



In [68]:
# ============================================================
# MODELO B2 — DISTRIBUIÇÃO DAS PREVISÕES POR CLASSE REAL
# ============================================================

analise_scores_b2 = (
    resultado_validacao_b2
    .groupby("status_real")["previsto"]
    .agg([
        "count",
        "min",
        "mean",
        "median",
        "max"
    ])
)

print(analise_scores_b2)

print("\nPercentis das previsões — RISCO")

print(
    resultado_validacao_b2.loc[
        resultado_validacao_b2["status_real"] == "Risco",
        "previsto"
    ].quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

print("\nPercentis das previsões — ESTABILIZADO")

print(
    resultado_validacao_b2.loc[
        resultado_validacao_b2["status_real"] == "Estabilizado",
        "previsto"
    ].quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

              count       min      mean    median       max
status_real                                                
Estabilizado   4098  0.419992  0.473341  0.480219  0.606251
Risco            90  0.445911  0.521434  0.524525  0.596913

Percentis das previsões — RISCO
0.10    0.487787
0.25    0.496006
0.50    0.524525
0.75    0.543380
0.90    0.560195
0.95    0.568282
0.99    0.594583
Name: previsto, dtype: float64

Percentis das previsões — ESTABILIZADO
0.10    0.435622
0.25    0.443443
0.50    0.480219
0.75    0.499355
0.90    0.509587
0.95    0.526671
0.99    0.553908
Name: previsto, dtype: float64


In [69]:
# ============================================================
# MODELO B2 — TESTE DE LIMIARES DE ALERTA ANTECIPADO
# ============================================================

from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

limiares = [
    0.45,
    0.47,
    0.48,
    0.49,
    0.50,
    0.51,
    0.52,
    0.53,
    0.54,
    0.55,
    0.56,
    0.57,
    0.58,
    0.59,
    0.60
]

resultados_limiares = []

y_real_risco = (
    resultado_validacao_b2["target_30min"] >= 0.60
).astype(int)

for limiar in limiares:

    y_alerta = (
        resultado_validacao_b2["previsto"] >= limiar
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_real_risco,
        y_alerta,
        labels=[0, 1]
    ).ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precisao = tp / (tp + fp) if (tp + fp) > 0 else 0

    especificidade = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )

    f1 = (
        2 * precisao * recall / (precisao + recall)
        if (precisao + recall) > 0
        else 0
    )

    resultados_limiares.append({
        "limiar_alerta": limiar,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "recall_risco": recall,
        "precisao_risco": precisao,
        "especificidade": especificidade,
        "f1_risco": f1
    })

comparacao_limiares = pd.DataFrame(
    resultados_limiares
)

comparacao_limiares

,limiar_alerta,TP,FP,FN,TN,recall_risco,precisao_risco,especificidade,f1_risco
0,0.45,88,2523,2,1575,0.977778,0.033704,0.384334,0.065161
1,0.47,85,2111,5,1987,0.944444,0.038707,0.484871,0.074366
2,0.48,84,2053,6,2045,0.933333,0.039307,0.499024,0.075438
3,0.49,75,1556,15,2542,0.833333,0.045984,0.620303,0.087159
4,0.50,62,979,28,3119,0.688889,0.059558,0.761103,0.109637
5,0.51,55,397,35,3701,0.611111,0.121681,0.903123,0.202952
6,0.52,46,258,44,3840,0.511111,0.151316,0.937042,0.233503
7,0.53,42,186,48,3912,0.466667,0.184211,0.954612,0.264151
8,0.54,26,110,64,3988,0.288889,0.191176,0.973158,0.230088
9,0.55,17,54,73,4044,0.188889,0.239437,0.986823,0.211180


In [70]:
# ============================================================
# COMPARAÇÃO MODELO B × MODELO B2 POR LIMIAR
# ============================================================

from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

limiares = [
    0.45, 0.47, 0.48, 0.49, 0.50,
    0.51, 0.52, 0.53, 0.54, 0.55,
    0.56, 0.57, 0.58, 0.59, 0.60
]

# O risco REAL continua sendo definido por GO >= 0.60
y_real_risco = (
    y_validacao_b >= 0.60
).astype(int)


def avaliar_limiares(nome_modelo, previsoes):

    resultados = []

    for limiar in limiares:

        y_alerta = (
            previsoes >= limiar
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_real_risco,
            y_alerta,
            labels=[0, 1]
        ).ravel()

        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0 else 0
        )

        precisao = (
            tp / (tp + fp)
            if (tp + fp) > 0 else 0
        )

        especificidade = (
            tn / (tn + fp)
            if (tn + fp) > 0 else 0
        )

        f1 = (
            2 * precisao * recall / (precisao + recall)
            if (precisao + recall) > 0 else 0
        )

        resultados.append({
            "modelo": nome_modelo,
            "limiar": limiar,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "recall": recall,
            "precisao": precisao,
            "especificidade": especificidade,
            "f1": f1
        })

    return pd.DataFrame(resultados)


comparacao_b = avaliar_limiares(
    "B",
    y_pred_b
)

comparacao_b2 = avaliar_limiares(
    "B2",
    y_pred_b2
)

comparacao_thresholds = pd.concat(
    [comparacao_b, comparacao_b2],
    ignore_index=True
)

comparacao_thresholds

,modelo,limiar,TP,FP,FN,TN,recall,precisao,especificidade,f1
0,B,0.45,88,2467,2,1631,0.977778,0.034442,0.397999,0.066541
1,B,0.47,85,2107,5,1991,0.944444,0.038777,0.485847,0.074496
2,B,0.48,85,2007,5,2091,0.944444,0.040631,0.510249,0.077910
3,B,0.49,70,1436,20,2662,0.777778,0.046481,0.649585,0.087719
4,B,0.50,62,970,28,3128,0.688889,0.060078,0.763299,0.110517
5,B,0.51,57,420,33,3678,0.633333,0.119497,0.897511,0.201058
6,B,0.52,43,216,47,3882,0.477778,0.166023,0.947291,0.246418
7,B,0.53,21,100,69,3998,0.233333,0.173554,0.975598,0.199052
8,B,0.54,13,36,77,4062,0.144444,0.265306,0.991215,0.187050
9,B,0.55,9,20,81,4078,0.100000,0.310345,0.995120,0.151261


In [71]:
# ============================================================
# COMPARAÇÃO B × B2 LADO A LADO
# ============================================================

comparacao_lado_a_lado = comparacao_thresholds.pivot(
    index="limiar",
    columns="modelo",
    values=[
        "TP",
        "FP",
        "FN",
        "recall",
        "precisao",
        "especificidade",
        "f1"
    ]
)

comparacao_lado_a_lado.round(4)


TP            FP            FN        recall         precisao  \
modelo     B    B2       B      B2     B    B2       B      B2        B   
limiar                                                                    
0.45    88.0  88.0  2467.0  2523.0   2.0   2.0  0.9778  0.9778   0.0344   
0.47    85.0  85.0  2107.0  2111.0   5.0   5.0  0.9444  0.9444   0.0388   
0.48    85.0  84.0  2007.0  2053.0   5.0   6.0  0.9444  0.9333   0.0406   
0.49    70.0  75.0  1436.0  1556.0  20.0  15.0  0.7778  0.8333   0.0465   
0.50    62.0  62.0   970.0   979.0  28.0  28.0  0.6889  0.6889   0.0601   
0.51    57.0  55.0   420.0   397.0  33.0  35.0  0.6333  0.6111   0.1195   
0.52    43.0  46.0   216.0   258.0  47.0  44.0  0.4778  0.5111   0.1660   
0.53    21.0  42.0   100.0   186.0  69.0  48.0  0.2333  0.4667   0.1736   
0.54    13.0  26.0    36.0   110.0  77.0  64.0  0.1444  0.2889   0.2653   
0.55     9.0  17.0    20.0    54.0  81.0  73.0  0.1000  0.1889   0.3103   
0.56     4.0  10.0    10.0    30.0  86.0  80.0  0.0444  0.1111   0.2857   
0.57     4.0   5.0     4.0    15.0  86.0  85.0  0.0444  0.0556   0.5000   
0.58     3.0   4.0     1.0     7.0  87.0  86.0  0.0333  0.0444   0.7500   
0.59     1.0   2.0     1.0     3.0  89.0  88.0  0.0111  0.0222   0.5000   
0.60     0.0   0.0     0.0     1.0  90.0  90.0  0.0000  0.0000   0.0000   

               especificidade              f1          
modelo      B2              B      B2       B      B2  
limiar                                                 
0.45    0.0337         0.3980  0.3843  0.0665  0.0652  
0.47    0.0387         0.4858  0.4849  0.0745  0.0744  
0.48    0.0393         0.5102  0.4990  0.0779  0.0754  
0.49    0.0460         0.6496  0.6203  0.0877  0.0872  
0.50    0.0596         0.7633  0.7611  0.1105  0.1096  
0.51    0.1217         0.8975  0.9031  0.2011  0.2030  
0.52    0.1513         0.9473  0.9370  0.2464  0.2335  
0.53    0.1842         0.9756  0.9546  0.1991  0.2642  
0.54    0.1912         0.9912  0.9732  0.1871  0.2301  
0.55    0.2394         0.9951  0.9868  0.1513  0.2112  
0.56    0.2500         0.9976  0.9927  0.0769  0.1538  
0.57    0.2500         0.9990  0.9963  0.0816  0.0909  
0.58    0.3636         0.9998  0.9983  0.0638  0.0792  
0.59    0.4000         0.9998  0.9993  0.0217  0.0421  
0.60    0.0000         1.0000  0.9998  0.0000  0.0000

In [72]:
# ============================================================
# EXPERIMENTO CONTROLADO DE PESOS
# Random Forest B — pesos 1x, 2x, 3x, 5x, 7x e 10x
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd
import time

pesos_teste = [1, 2, 3, 5, 7, 10]

resultados_pesos = []
previsoes_pesos = {}
modelos_pesos = {}

for peso_risco in pesos_teste:

    print(f"\n{'='*55}")
    print(f"TESTANDO PESO DE RISCO = {peso_risco}x")
    print(f"{'='*55}")

    # Peso maior apenas para registros de risco no TREINO
    sample_weights = np.where(
        y_treino_b >= 0.60,
        float(peso_risco),
        1.0
    )

    modelo = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )

    # ----------------------------
    # Treinamento
    # ----------------------------
    inicio_treino = time.perf_counter()

    modelo.fit(
        X_treino_processado_b,
        y_treino_b,
        sample_weight=sample_weights
    )

    tempo_treino = time.perf_counter() - inicio_treino

    # ----------------------------
    # Predição
    # ----------------------------
    inicio_predicao = time.perf_counter()

    y_pred = modelo.predict(
        X_validacao_processado_b
    )

    tempo_predicao = time.perf_counter() - inicio_predicao

    # ----------------------------
    # Métricas de regressão
    # ----------------------------
    mae = mean_absolute_error(
        y_validacao_b,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_validacao_b,
            y_pred
        )
    )

    r2 = r2_score(
        y_validacao_b,
        y_pred
    )

    # Guarda modelos e previsões
    modelos_pesos[peso_risco] = modelo
    previsoes_pesos[peso_risco] = y_pred

    resultados_pesos.append({
        "peso_risco": peso_risco,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "tempo_treino_s": tempo_treino,
        "tempo_predicao_s": tempo_predicao,
        "pred_media": np.mean(y_pred),
        "pred_max": np.max(y_pred)
    })

resultados_regressao_pesos = pd.DataFrame(
    resultados_pesos
)

print("\n\nRESULTADO FINAL")
display(
    resultados_regressao_pesos.round(6)
)


TESTANDO PESO DE RISCO = 1x

TESTANDO PESO DE RISCO = 2x

TESTANDO PESO DE RISCO = 3x

TESTANDO PESO DE RISCO = 5x

TESTANDO PESO DE RISCO = 7x

TESTANDO PESO DE RISCO = 10x


RESULTADO FINAL


,peso_risco,MAE,RMSE,R2,tempo_treino_s,tempo_predicao_s,pred_media,pred_max
0,1,0.039028,0.049504,0.274513,4.163213,0.063463,0.472726,0.593205
1,2,0.039162,0.049657,0.270019,3.011357,0.045254,0.473447,0.603065
2,3,0.039163,0.049657,0.270032,3.773969,0.067781,0.473625,0.605427
3,5,0.039341,0.049892,0.263084,3.888397,0.044098,0.474374,0.606251
4,7,0.039334,0.049923,0.262172,2.881715,0.044370,0.474592,0.599575
5,10,0.039428,0.050018,0.259364,2.912460,0.045427,0.475378,0.598000


In [73]:
# ============================================================
# C3 — COMPARAÇÃO FINAL: PESO × LIMIAR DE ALERTA
# ============================================================

from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

limiares = [
    0.45, 0.47, 0.48, 0.49, 0.50,
    0.51, 0.52, 0.53, 0.54, 0.55,
    0.56, 0.57, 0.58, 0.59, 0.60
]

y_real = (y_validacao_b >= 0.60).astype(int)

resultados_c3 = []

for peso, previsoes in previsoes_pesos.items():

    for limiar in limiares:

        y_pred_classe = (
            previsoes >= limiar
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_real,
            y_pred_classe,
            labels=[0, 1]
        ).ravel()

        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0 else 0
        )

        precisao = (
            tp / (tp + fp)
            if (tp + fp) > 0 else 0
        )

        especificidade = (
            tn / (tn + fp)
            if (tn + fp) > 0 else 0
        )

        f1 = (
            2 * precisao * recall / (precisao + recall)
            if (precisao + recall) > 0 else 0
        )

        resultados_c3.append({
            "peso": peso,
            "limiar": limiar,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "recall": recall,
            "precisao": precisao,
            "especificidade": especificidade,
            "f1": f1
        })

df_c3 = pd.DataFrame(resultados_c3)

print("Total de combinações:", len(df_c3))

display(df_c3.round(4))

Total de combinações: 90


,peso,limiar,TP,FP,FN,TN,recall,precisao,especificidade,f1
0,1,0.45,88,2467,2,1631,0.9778,0.0344,0.3980,0.0665
1,1,0.47,85,2107,5,1991,0.9444,0.0388,0.4858,0.0745
2,1,0.48,85,2007,5,2091,0.9444,0.0406,0.5102,0.0779
3,1,0.49,70,1436,20,2662,0.7778,0.0465,0.6496,0.0877
4,1,0.50,62,970,28,3128,0.6889,0.0601,0.7633,0.1105
...,...,...,...,...,...,...,...,...,...,...
85,10,0.56,11,40,79,4058,0.1222,0.2157,0.9902,0.1560
86,10,0.57,4,17,86,4081,0.0444,0.1905,0.9959,0.0721
87,10,0.58,4,5,86,4093,0.0444,0.4444,0.9988,0.0808
88,10,0.59,2,2,88,4096,0.0222,0.5000,0.9995,0.0426


In [74]:
# ============================================================
# C3.1 — MELHOR LIMIAR OBSERVADO PARA CADA PESO
# ============================================================

indices_melhores = (
    df_c3
    .groupby("peso")["f1"]
    .idxmax()
)

melhores_por_peso = (
    df_c3
    .loc[indices_melhores]
    .sort_values("peso")
    .reset_index(drop=True)
)

# Adiciona métricas de regressão
melhores_por_peso = melhores_por_peso.merge(
    resultados_regressao_pesos[
        ["peso_risco", "MAE", "RMSE", "R2"]
    ],
    left_on="peso",
    right_on="peso_risco",
    how="left"
)

melhores_por_peso = melhores_por_peso.drop(
    columns=["peso_risco"]
)

display(
    melhores_por_peso[
        [
            "peso",
            "limiar",
            "TP",
            "FP",
            "FN",
            "recall",
            "precisao",
            "especificidade",
            "f1",
            "MAE",
            "RMSE",
            "R2"
        ]
    ].round(4)
)

,peso,limiar,TP,FP,FN,recall,precisao,especificidade,f1,MAE,RMSE,R2
0,1,0.52,43,216,47,0.4778,0.1660,0.9473,0.2464,0.0390,0.0495,0.2745
1,2,0.52,44,231,46,0.4889,0.1600,0.9436,0.2411,0.0392,0.0497,0.2700
2,3,0.53,35,159,55,0.3889,0.1804,0.9612,0.2465,0.0392,0.0497,0.2700
3,5,0.53,42,186,48,0.4667,0.1842,0.9546,0.2642,0.0393,0.0499,0.2631
4,7,0.54,31,119,59,0.3444,0.2067,0.9710,0.2583,0.0393,0.0499,0.2622
5,10,0.53,47,232,43,0.5222,0.1685,0.9434,0.2547,0.0394,0.0500,0.2594
